## this notebook is used for "Cohort Builder"-derived cohort building

adapted from notebook:

"1 - CB case phenotype algorithm V2 SQL query: CB cohort building"

in workspace "Hypothyroidism genomics v7"


In [ ]:
# !pip install polars
# !pip install matplotlib-venn

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import json
import re
from google.cloud import bigquery
import polars as pl
import gcsfs
from matplotlib_venn import venn2

In [ ]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)


# Get workspace info
workspace = wb("workspace", "describe")
GOOGLE_CLOUD_PROJECT = workspace["googleProjectId"]

# Get resources
resources = wb("resource", "list")

# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]

if not bucket_resources:
    raise ValueError("No matching bucket found")

WORKSPACE_BUCKET = f"gs://{bucket_resources[0]['bucketName']}"

# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]

cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]

if not cdr_resources:
    raise ValueError("No matching CDR dataset found")

WORKSPACE_CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
#get variables
version = WORKSPACE_CDR
bucket = WORKSPACE_BUCKET
cohort = "allofus"
google_project_id = GOOGLE_CLOUD_PROJECT

## helper funx

In [ ]:
def polars_gbq(query):
    """
    Take a SQL query and return result as polars dataframe
    :param query: BigQuery SQL query
    :return: polars dataframe
    """
    client = bigquery.Client()
    query_job = client.query(query)
    rows = query_job.result()
    df = pl.from_arrow(rows.to_arrow())

    return df

In [ ]:
#query cases
# adapted from
# 1 - CB case phenotype algorithm V2 SQL query in workspace "Hypothyroidism genomics v7"
cb_case_v2_q = f"""
WITH incl_concept_criteria AS (
    -- Block A: inclusion concepts (137520, 138384, 140673, 135215, 4281109, 133444)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (137520, 138384, 140673, 135215, 4281109, 133444)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

lab_criteria_1 AS (
    -- Block B: first lab-value threshold set
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE (
        (concept_id = 3009201 AND is_standard = 1 AND value_as_number >= 5.0)
        OR (concept_id = 3019170 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (4328749, 45878745)))
        OR (concept_id = 3019762 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (45876384, 4328749)))
        OR (concept_id = 3008598 AND is_standard = 1 AND value_as_number <= 0.5)
        OR (concept_id = 3000551 AND is_standard = 1 AND value_as_number <= 0.5)
        OR (concept_id = 4197602 AND is_standard = 1 AND value_as_number >= 5.0)
        OR (concept_id = 4196969 AND is_standard = 1
            AND (value_as_number <= 0.5 OR value_as_concept_id IN (45881666)))
    )
),

ancestor_criteria_1 AS (
    -- Block C: ancestor-based criteria on (1505346, 1501700), first occurrence
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT ca.descendant_id
        FROM {version}.cb_criteria_ancestor ca
        JOIN (
            SELECT DISTINCT c.concept_id
            FROM {version}.cb_criteria c
            JOIN (
                SELECT CAST(cr.id AS STRING) AS id
                FROM {version}.cb_criteria cr
                WHERE concept_id IN (1505346, 1501700)
                  AND full_text LIKE '%_rank1]%'
            ) a
            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                OR c.path LIKE CONCAT('%.', a.id)
                OR c.path LIKE CONCAT(a.id, '.%')
                OR c.path = a.id)
            WHERE is_standard = 1 AND is_selectable = 1
        ) b
        ON ca.ancestor_id = b.concept_id
    )
    AND is_standard = 1
),

lab_criteria_2 AS (
    -- Block B (nested variant): adds 45876384 to 3019170, adds 4267416 to 3008598
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE (
        (concept_id = 3009201 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (45876384)))
        OR (concept_id = 3019170 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (4328749, 45876384)))
        OR (concept_id = 3019762 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (45876384, 4328749)))
        OR (concept_id = 3008598 AND is_standard = 1
            AND (value_as_number <= 0.5 OR value_as_concept_id IN (4267416, 45881666)))
        OR (concept_id = 3000551 AND is_standard = 1 AND value_as_number <= 0.5)
        OR (concept_id = 4197602 AND is_standard = 1 AND value_as_number >= 5.0)
        OR (concept_id = 4196969 AND is_standard = 1
            AND (value_as_number <= 0.5 OR value_as_concept_id IN (45881666)))
    )
),

lab_criteria_3 AS (
    -- Block D component (temp1 source): same shape as lab_criteria_2
    SELECT person_id, visit_occurrence_id, entry_date
    FROM {version}.cb_search_all_events
    WHERE (
        (concept_id = 3009201 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (45876384)))
        OR (concept_id = 3019170 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (4328749)))
        OR (concept_id = 3019762 AND is_standard = 1
            AND (value_as_number >= 5.0 OR value_as_concept_id IN (45876384, 4328749)))
        OR (concept_id = 3008598 AND is_standard = 1
            AND (value_as_number <= 0.5 OR value_as_concept_id IN (4267416, 45881666)))
        OR (concept_id = 3000551 AND is_standard = 1 AND value_as_number <= 0.5)
        OR (concept_id = 4197602 AND is_standard = 1 AND value_as_number >= 5.0)
        OR (concept_id = 4196969 AND is_standard = 1
            AND (value_as_number <= 0.5 OR value_as_concept_id IN (45881666)))
    )
    AND person_id IN (SELECT person_id FROM lab_criteria_2)
),

ancestor_criteria_2 AS (
    -- Block C repeated (used to build temp2 event set)
    SELECT DISTINCT ca.descendant_id AS concept_id
    FROM {version}.cb_criteria_ancestor ca
    JOIN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (1505346, 1501700)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    ) b
    ON ca.ancestor_id = b.concept_id
),

ancestor_criteria_3 AS (
    -- Block C repeated again (innermost re-check feeding temp2's person filter)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (SELECT concept_id FROM ancestor_criteria_2)
    AND is_standard = 1
),

lab_criteria_4 AS (
    -- Block D component (temp2 source): base lab pattern, gated by ancestor_criteria_3
    SELECT person_id, visit_occurrence_id, entry_date
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT ca.descendant_id
        FROM {version}.cb_criteria_ancestor ca
        JOIN (
            SELECT DISTINCT c.concept_id
            FROM {version}.cb_criteria c
            JOIN (
                SELECT CAST(cr.id AS STRING) AS id
                FROM {version}.cb_criteria cr
                WHERE concept_id IN (1505346, 1501700)
                  AND full_text LIKE '%_rank1]%'
            ) a
            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                OR c.path LIKE CONCAT('%.', a.id)
                OR c.path LIKE CONCAT(a.id, '.%')
                OR c.path = a.id)
            WHERE is_standard = 1 AND is_selectable = 1
        ) b
        ON ca.ancestor_id = b.concept_id
    )
    AND is_standard = 1
    AND person_id IN (SELECT person_id FROM ancestor_criteria_3)
),

temp1 AS (
    SELECT person_id, visit_occurrence_id, entry_date
    FROM lab_criteria_3
    UNION ALL
    SELECT person_id, visit_occurrence_id, entry_date
    FROM lab_criteria_4
),

temp2 AS (
    -- mirrors temp1's construction (same union) per original nested structure
    SELECT person_id, visit_occurrence_id, entry_date
    FROM lab_criteria_3
    UNION ALL
    SELECT person_id, visit_occurrence_id, entry_date
    FROM lab_criteria_4
),

temporal_join AS (
    -- Block D: temp1 entry_date >= temp2 entry_date + 90 days
    SELECT DISTINCT temp1.person_id
    FROM temp1
    JOIN temp2
    ON temp1.person_id = temp2.person_id
    AND temp1.entry_date >= DATE_ADD(temp2.entry_date, INTERVAL 90 DAY)
),

excl_concept_criteria AS (
    -- Block E part 1: exclusion concepts (4241917, 134618, 36676291, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (4241917, 134618, 36676291, 199775, 4030049, 137820,
                                  138717, 24612, 37016342, 140062, 134312, 133727,
                                  140976, 4231548, 135778, 28127, 4178976)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

excl_ancestor_criteria AS (
    -- Block E part 2: exclusion ancestor set (1554072, 1309944, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT ca.descendant_id
        FROM {version}.cb_criteria_ancestor ca
        JOIN (
            SELECT DISTINCT c.concept_id
            FROM {version}.cb_criteria c
            JOIN (
                SELECT CAST(cr.id AS STRING) AS id
                FROM {version}.cb_criteria cr
                WHERE concept_id IN (1554072, 1309944, 19124477, 740910, 751246,
                                      1504620, 767410, 19127669)
                  AND full_text LIKE '%_rank1]%'
            ) a
            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                OR c.path LIKE CONCAT('%.', a.id)
                OR c.path LIKE CONCAT(a.id, '.%')
                OR c.path = a.id)
            WHERE is_standard = 1 AND is_selectable = 1
        ) b
        ON ca.ancestor_id = b.concept_id
    )
    AND is_standard = 1
),

excl_explicit_concepts AS (
    -- Block E part 3: large explicit exclusion concept list
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        2110385, 42737622, 2211849, 2211841, 2211845, 2211880, 2211862, 2110376,
        2211860, 2110369, 2211872, 2211840, 2211838, 2211871, 2110375, 2211906,
        42737621, 2211847, 2110365, 2211876, 2211848, 2211844, 2211846, 42737623,
        2211866, 2211834, 2211853, 2211835, 2110383, 2211894, 2211869, 2211837,
        2211891, 2110384, 2211857, 2211868, 2211870, 2211897, 2110374, 2211839,
        2211895, 2110368, 2211905, 2211836, 42737624, 2110370, 2211831, 2211842,
        2211861, 2211859, 2211882, 2211884, 2211850, 2110371, 2110372, 2211865,
        2211878, 2110373, 2211832, 2211864, 2211885, 2211877, 2211893, 2211881,
        2211883, 2211858, 2211867, 2211852, 2211843, 2211907, 2211919, 2110367,
        2211833
    )
    AND is_standard = 1
),

excl_concept_criteria_2 AS (
    -- Block E part 4: second exclusion concept-list block (444094, 436477, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (444094, 436477, 4239301, 200153, 193525, 4307820,
                                  432969, 432695, 43530950, 4118058, 139895, 437611)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

lab_criteria_5 AS (
    -- Block F: final lab-value threshold set (3018171, 3003191, 3018954, 3038136)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE (
        (concept_id = 3018171 AND is_standard = 1 AND value_as_number >= 25.0)
        OR (concept_id = 3003191 AND is_standard = 1
            AND (value_as_number >= 25.0 OR value_as_concept_id IN (45884084, 4328749)))
        OR (concept_id = 3018954 AND is_standard = 1
            AND value_as_concept_id IN (9191, 45878745, 45884084))
        OR (concept_id = 3038136 AND is_standard = 1
            AND (value_as_number >= 25.0 OR value_as_concept_id IN (45876384)))
    )
),

final_cohort AS (
    SELECT person_id
    FROM incl_concept_criteria
    WHERE person_id IN (SELECT person_id FROM lab_criteria_1)
      AND person_id IN (SELECT person_id FROM ancestor_criteria_1)
      AND person_id IN (SELECT person_id FROM temporal_join)
      AND person_id NOT IN (
          SELECT person_id FROM excl_concept_criteria
          UNION ALL
          SELECT person_id FROM excl_ancestor_criteria
          UNION ALL
          SELECT person_id FROM excl_explicit_concepts
          UNION ALL
          SELECT person_id FROM excl_concept_criteria_2
          UNION ALL
          SELECT person_id FROM lab_criteria_5
      )
)

SELECT person.person_id
FROM {version}.person AS person
WHERE person.person_id IN (SELECT person_id FROM final_cohort)
"""

cb_case_v2 = polars_gbq(cb_case_v2_q)

In [ ]:
cb_case_v2 = cb_case_v2.with_columns(pl.lit(1).alias("Hypothyroidism"))

In [ ]:
#query controls
# "1 - cb_phenotype_control_v2" (person domain, All of Us Controlled Tier v6)
cb_control_v2_q = f"""
WITH lab_criteria_1 AS (
    -- Inclusion: TSH/related labs within normal range (0.5-5.0), TSH-specific 0.5-1.2
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE (
        (concept_id = 3009201 AND is_standard = 1 AND value_as_number BETWEEN 0.5 AND 5.0)
        OR (concept_id = 3019170 AND is_standard = 1 AND value_as_number BETWEEN 0.5 AND 5.0)
        OR (concept_id = 3016991 AND is_standard = 1 AND value_as_number BETWEEN 0.5 AND 1.2)
    )
),

excl_concept_criteria_1a AS (
    -- Exclusion part 1: concept list (199775, 134619, 137820, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (199775, 134619, 137820, 133424, 24612, 36717452,
                                  140062, 134312, 4131812, 133727, 140976, 132583,
                                  4113641, 135778, 28127)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

excl_explicit_concepts_1a AS (
    -- Exclusion part 2: large explicit concept list
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        2110385, 42737622, 2211849, 2211841, 2211845, 2211880, 2211862, 2110376,
        2211860, 2110369, 2211872, 2211840, 2211838, 2211871, 2110375, 2211906,
        42737621, 2211847, 2110365, 2211876, 2211848, 2211844, 2211846, 42737623,
        2211866, 2211834, 2211853, 2211835, 2110383, 2211894, 2211869, 2211837,
        2211891, 2110384, 2211857, 2211868, 2211870, 2211897, 2110374, 2211839,
        2211895, 2110368, 2211905, 2211836, 42737624, 2211831, 2211842, 2211861,
        2211859, 2211882, 2211884, 2211850, 2110371, 2110372, 2211865, 2211878,
        2110373, 2211832, 2211864, 2211885, 2211877, 2211893, 2211881, 2211883,
        2211858, 2211867, 2211852, 2211843, 2211907, 2211919, 2110367, 2211833
    )
    AND is_standard = 1
),

excl_ancestor_criteria_1a AS (
    -- Exclusion part 3: ancestor-based set (1554072, 1309944, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT ca.descendant_id
        FROM {version}.cb_criteria_ancestor ca
        JOIN (
            SELECT DISTINCT c.concept_id
            FROM {version}.cb_criteria c
            JOIN (
                SELECT CAST(cr.id AS STRING) AS id
                FROM {version}.cb_criteria cr
                WHERE concept_id IN (1554072, 1309944, 19124477, 740910, 751246, 1504620)
                  AND full_text LIKE '%_rank1]%'
            ) a
            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                OR c.path LIKE CONCAT('%.', a.id)
                OR c.path LIKE CONCAT(a.id, '.%')
                OR c.path = a.id)
            WHERE is_standard = 1 AND is_selectable = 1
        ) b
        ON ca.ancestor_id = b.concept_id
    )
    AND is_standard = 1
),

excl_concept_criteria_2a AS (
    -- Exclusion part 4: concept list (135772, 133728, 141253, ...), first occurrence
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (135772, 133728, 141253, 76685, 4311117, 138384,
                                  133444, 4113641)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

excl_concept_criteria_2b AS (
    -- Exclusion part 4 repeated: same concept list (135772, 133728, 141253, ...), second occurrence
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (135772, 133728, 141253, 76685, 4311117, 138384,
                                  133444, 4113641)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

excl_concept_criteria_1b AS (
    -- Innermost re-check, exclusion part 1 repeated (199775, 134619, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM {version}.cb_criteria c
        JOIN (
            SELECT CAST(cr.id AS STRING) AS id
            FROM {version}.cb_criteria cr
            WHERE concept_id IN (199775, 134619, 137820, 133424, 24612, 36717452,
                                  140062, 134312, 4131812, 133727, 140976, 132583,
                                  4113641, 135778, 28127)
              AND full_text LIKE '%_rank1]%'
        ) a
        ON (c.path LIKE CONCAT('%.', a.id, '.%')
            OR c.path LIKE CONCAT('%.', a.id)
            OR c.path LIKE CONCAT(a.id, '.%')
            OR c.path = a.id)
        WHERE is_standard = 1 AND is_selectable = 1
    )
    AND is_standard = 1
),

excl_explicit_concepts_1b AS (
    -- Innermost re-check, exclusion part 2 repeated
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        2110385, 42737622, 2211849, 2211841, 2211845, 2211880, 2211862, 2110376,
        2211860, 2110369, 2211872, 2211840, 2211838, 2211871, 2110375, 2211906,
        42737621, 2211847, 2110365, 2211876, 2211848, 2211844, 2211846, 42737623,
        2211866, 2211834, 2211853, 2211835, 2110383, 2211894, 2211869, 2211837,
        2211891, 2110384, 2211857, 2211868, 2211870, 2211897, 2110374, 2211839,
        2211895, 2110368, 2211905, 2211836, 42737624, 2211831, 2211842, 2211861,
        2211859, 2211882, 2211884, 2211850, 2110371, 2110372, 2211865, 2211878,
        2110373, 2211832, 2211864, 2211885, 2211877, 2211893, 2211881, 2211883,
        2211858, 2211867, 2211852, 2211843, 2211907, 2211919, 2110367, 2211833
    )
    AND is_standard = 1
),

excl_ancestor_criteria_1b AS (
    -- Innermost re-check, exclusion part 3 repeated (1554072, 1309944, ...)
    SELECT DISTINCT person_id
    FROM {version}.cb_search_all_events
    WHERE concept_id IN (
        SELECT DISTINCT ca.descendant_id
        FROM {version}.cb_criteria_ancestor ca
        JOIN (
            SELECT DISTINCT c.concept_id
            FROM {version}.cb_criteria c
            JOIN (
                SELECT CAST(cr.id AS STRING) AS id
                FROM {version}.cb_criteria cr
                WHERE concept_id IN (1554072, 1309944, 19124477, 740910, 751246, 1504620)
                  AND full_text LIKE '%_rank1]%'
            ) a
            ON (c.path LIKE CONCAT('%.', a.id, '.%')
                OR c.path LIKE CONCAT('%.', a.id)
                OR c.path LIKE CONCAT(a.id, '.%')
                OR c.path = a.id)
            WHERE is_standard = 1 AND is_selectable = 1
        ) b
        ON ca.ancestor_id = b.concept_id
    )
    AND is_standard = 1
),

excl_set_outer AS (
    -- Outer-level exclusion set: parts 1-4 combined, all NOT IN members
    SELECT person_id FROM excl_concept_criteria_1a
    UNION ALL
    SELECT person_id FROM excl_explicit_concepts_1a
    UNION ALL
    SELECT person_id FROM excl_ancestor_criteria_1a
    UNION ALL
    SELECT person_id FROM excl_concept_criteria_2a
),

excl_set_inner AS (
    -- Inner-level exclusion set: parts 1-4 repeated, all NOT IN members
    SELECT person_id FROM excl_concept_criteria_1b
    UNION ALL
    SELECT person_id FROM excl_explicit_concepts_1b
    UNION ALL
    SELECT person_id FROM excl_ancestor_criteria_1b
    UNION ALL
    SELECT person_id FROM excl_concept_criteria_2b
),

final_cohort AS (
    SELECT person_id
    FROM lab_criteria_1
    WHERE person_id NOT IN (SELECT person_id FROM excl_set_outer)
      AND person_id NOT IN (SELECT person_id FROM excl_set_inner)
)

SELECT person.person_id
FROM {version}.person AS person
WHERE person.person_id IN (SELECT person_id FROM final_cohort)
"""

cb_control_v1 = polars_gbq(cb_control_v2_q)

In [ ]:
cb_control_v1 = cb_control_v1.with_columns(pl.lit(0).alias("Hypothyroidism"))

In [ ]:
cb_v2 = pl.concat([cb_case_v2, cb_control_v1])

In [ ]:
#print case control counts
cb_v2["Hypothyroidism"].value_counts()

In [ ]:
cb_v2 = cb_v2.join(
    polars_gbq(f'''SELECT DISTINCT person_id, sex_at_birth, age_at_cdr FROM {version}.cb_search_person'''),
    how="left",
    on="person_id"
)

In [ ]:
cb_v2 = cb_v2.filter(pl.col("sex_at_birth").is_in(["Female", "Male"]))

In [ ]:
print(cb_v2["sex_at_birth"].value_counts())

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open('gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv') as f:
    ancestry_df = pl.read_csv(f, separator='\t').select(['research_id', 'ancestry_pred'])

cb_v2 = cb_v2.join(
    ancestry_df,
    how='left',
    left_on='person_id',
    right_on='research_id'
)

In [ ]:
cb_v2 = cb_v2.drop_nulls()
cb_v2["Hypothyroidism"].value_counts()

In [ ]:
fs = gcsfs.GCSFileSystem(requester_pays=True)
with fs.open(f'{bucket}/hypothyroid_data/cb_v2_phenotype_covars.tsv', 'w') as f:
    cb_v2.write_csv(f, separator='\t')

In [ ]:
huan = pd.read_csv(f'{bucket}/hypothyroid_data/huan_phenotype_v4.csv')
huan

In [ ]:
# venn2([
#     set(cb_v2.query('Hypothyroidism == 1').person_id.unique()),
#     set(huan.query('hypothyroidism == 1').person_id.unique())
#         ], ('CB_V2','HUAN_V4'))

In [ ]:
venn2([
    set(cb_v2.query('Hypothyroidism == 0').person_id.unique()),
    set(huan.query('hypothyroidism == 0').person_id.unique())
        ],
('CB_V2','HUAN_V4'))